# Fart og akselerasjon fra måledata

Fra en video eller en bevegelsessensor får vi posisjonen til en gjenstand ved ulike tidspunkter. Men hvordan finner vi farten og akselerasjonen? Her lar vi Python regne ut stigningstall mellom målepunktene, og bruker det på en sprettball.

Oppgavene står på nettsiden: [Fart og akselerasjon fra måledata](https://lektorbo.no/programmering/python-bevegelse-maledata.html). Kjør cellene i rekkefølge med **Shift + Enter**.

> **Lagring:** Endringene dine lagres bare i denne nettleseren. Vil du ta vare på arbeidet eller levere det, velger du *File → Download*.

## Data

Fila har to kolonner: tiden i sekunder og høyden til ballen i meter, 30 målinger per sekund. Ballen slippes fra 0,65 m og spretter flere ganger. Dataene er laget med en modell med sprettall 0,80 og 1 mm tilfeldig målestøy, slik at de ligner data fra videoanalyse. De er ikke ekte målinger. Bytt gjerne ut fila med egne data fra Tracker.

## Steg 1: Posisjon, fart og akselerasjon

Farten i punkt i regnes ut som stigningstallet mellom naboene: v<sub>i</sub> = (s<sub>i+1</sub> − s<sub>i−1</sub>) / (t<sub>i+1</sub> − t<sub>i−1</sub>). Dette kalles en sentraldifferanse. Akselerasjonen finner vi på samme måte fra farten.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt("sprettball.csv", delimiter=",", skiprows=1)
t = data[:, 0]
s = data[:, 1]
n = len(t)

# Farten: sentraldifferanse, altså stigningstallet mellom naboene
v = np.zeros(n)
for i in range(1, n-1):
    v[i] = (s[i+1] - s[i-1]) / (t[i+1] - t[i-1])

# Akselerasjonen: samme metode brukt på farten
a = np.zeros(n)
for i in range(1, n-1):
    a[i] = (v[i+1] - v[i-1]) / (t[i+1] - t[i-1])

plt.figure(figsize=(8, 7))
plt.subplot(3, 1, 1)
plt.plot(t, s, ".-")
plt.ylabel("$s$ / m")
plt.grid()
plt.subplot(3, 1, 2)
plt.plot(t[1:n-1], v[1:n-1], ".-")
plt.ylabel("$v$ / (m/s)")
plt.grid()
plt.subplot(3, 1, 3)
plt.plot(t[2:n-2], a[2:n-2], ".-")
plt.xlabel("$t$ / s")
plt.ylabel("$a$ / (m/s$^2$)")
plt.grid()
plt.tight_layout()
plt.show()

## Steg 2: Tyngdeakselerasjonen fra det første fallet

Den bruker variablene fra steg 1, som cellene i en Jupyter-notatbok. Skriver du i en vanlig .py-fil, legger du koden nederst. Vi tilpasser en andregradsfunksjon til fallet og sammenligner med akselerasjonen fra steg 1.

In [ ]:
# Velg tidsrommet for det første fallet. Se på grafen og juster grensene.
fall = (t >= 0.13) & (t <= 0.43)
koeff = np.polyfit(t[fall], s[fall], 2)
print(f"Andregradsfunksjon: s = {koeff[0]:.3f} t² + {koeff[1]:.3f} t + {koeff[2]:.3f}")
print(f"Akselerasjon: {2*koeff[0]:.2f} m/s²")

# Gjennomsnittlig akselerasjon fra den numeriske deriveringen i samme tidsrom
print(f"Middelverdi av a i tidsrommet: {np.mean(a[fall]):.2f} m/s²")

## Steg 3: Hvor mye energi forsvinner i hvert sprett?

In [ ]:
# Finn toppunktene etter hvert sprett
topper = []
for i in range(1, n-1):
    if s[i] > s[i-1] and s[i] >= s[i+1] and s[i] > 0.05 and t[i] > 0.5:
        topper.append(s[i])
print("Toppunkter:", [round(float(h), 3) for h in topper])
for i in range(1, len(topper)):
    print(f"Forholdet mellom topp {i+1} og topp {i}: {topper[i]/topper[i-1]:.2f}")

## Din egen kode

Bruk cellen under til oppgavene. Du kan lage flere celler med **+** i verktøylinja.